# pick_place: motion primitives and chess-move sequences

Library notebook for `gambit_arm`. Builds every arm motion out of a few primitives
(move_to_hover, descend, grasp, lift, place/release, park), composes them into `pick`,
`place`, `move_piece` and `capture_to_graveyard`, and expands a chess move (including
castling, en passant and promotion) into those. A trajectory is a plain list of waypoints;
`plan_joint_trajectory` turns it into joint targets with IK and checks each against safety.
Execution backends (real servos, Gazebo controller, LeRobot) are TODO.

Milestone: *Hardware and execution* (pick-and-place success 55-60 % first attempt, 95 % with
verify-and-retry, see run_pick_place_trials.ipynb).

**TODO**
- [x] Cartesian waypoint primitives from board.yaml (hover height, grasp height fraction)
- [x] pick / place / move_piece / capture_to_graveyard, chess move -> sequence
- [x] IK + safety check per waypoint (`plan_joint_trajectory`), dry-run backend
- [x] joint-space interpolation (`densify`) with every step safety-checked
- [ ] straight-line descend/lift in Cartesian space, timing from safety.yaml velocity limits
- [ ] backends: `ServoBackend` (servo_bus.ipynb), `GazeboBackend` (FollowJointTrajectory on
      /arm_controller and /gripper_controller, see docs/gazebo.md), `LeRobotBackend`
- [ ] promotion with a spare queen from the graveyard; under-promotion
- [x] approach tilt chosen per waypoint from a candidate list (`solve_waypoint`)
- [ ] rank 1 (c1-f1) hover is out of reach with the current limits, see the reachability note
- [ ] finger orientation (wrist_roll) so the jaws close across the piece, not along a file
- [ ] tune descend speed and grasp pause on the real arm

In [ ]:
import math
from dataclasses import dataclass, field
import numpy as np

%run ./safety.ipynb

## Waypoints

A waypoint is either Cartesian (`xyz` of the gripper tip, plus a tilt from vertical) or a
joint-space pose (`q`, used for park). `gripper` is the gripper joint target to hold while
moving to it (board.yaml `gripper_open` / `gripper_closed`). `slow` marks segments near the
pieces (descend, lift) that must use the reduced Cartesian speed.

In [ ]:
ARM = BOARD["arm"]

@dataclass
class Waypoint:
    name: str
    xyz: np.ndarray = None
    q: np.ndarray = None
    gripper: float = ARM["gripper_open"]
    tilt_deg: float = 0.0
    slow: bool = False

    def __post_init__(self):
        assert (self.xyz is None) != (self.q is None), "give xyz or q, not both"

## Primitives

Each returns a list of waypoints. `target` is a square name or a world point (graveyard
slot). The gripper state is carried along: e.g. `lift` after `grasp` keeps it closed.

In [ ]:
def move_to_hover(target, gripper=ARM["gripper_open"]):
    return [Waypoint(f"hover {target if isinstance(target, str) else 'pt'}", xyz=hover_point(target), gripper=gripper)]

def descend(target, kind, gripper=ARM["gripper_open"]):
    return [Waypoint("descend", xyz=grasp_point(target, kind), gripper=gripper, slow=True)]

def grasp(target, kind):
    """Close the gripper in place at the grasp point."""
    return [Waypoint("grasp", xyz=grasp_point(target, kind), gripper=ARM["gripper_closed"], slow=True)]

def release(target, kind):
    return [Waypoint("release", xyz=grasp_point(target, kind), gripper=ARM["gripper_open"], slow=True)]

def lift(target, gripper):
    return [Waypoint("lift", xyz=hover_point(target), gripper=gripper, slow=True)]

def park(gripper=ARM["gripper_open"]):
    return [Waypoint("park", q=np.array(ARM["park"]), gripper=gripper)]

## Composite motions

In [ ]:
def pick(target, kind):
    """Hover, descend open, close, lift. Ends at hover holding the piece."""
    return (move_to_hover(target) + descend(target, kind) + grasp(target, kind)
            + lift(target, ARM["gripper_closed"]))

def place(target, kind):
    """Hover holding the piece, descend, open, lift. Ends at hover, open."""
    closed = ARM["gripper_closed"]
    return (move_to_hover(target, closed) + descend(target, kind, closed) + release(target, kind)
            + lift(target, ARM["gripper_open"]))

def move_piece(src, dst, kind):
    return pick(src, kind) + place(dst, kind)

def capture_to_graveyard(square, kind, slot):
    """Remove the piece on `square` to graveyard slot `slot` (placed on the table, z = 0)."""
    return pick(square, kind) + place(graveyard_slot(slot), kind)

## Chess move -> motion sequence

Order matters: clear the captured piece first, then move the mover. Castling moves the king
then the rook; en passant removes the pawn *behind* the destination square. Promotion would
swap the pawn for a spare queen from the graveyard (TODO). Uses python-chess (imported here
so the rest of the notebook does not need it).

In [ ]:
def plan_chess_move(board, move, next_grave_slot=0):
    """board: chess.Board BEFORE the move; move: chess.Move.
    Returns (waypoints, slots_used)."""
    import chess
    sq = chess.square_name
    mover = board.piece_at(move.from_square)
    if mover is None:
        raise ValueError(f"no piece on {sq(move.from_square)}")
    kind = mover.symbol().upper()
    wps, slot = [], next_grave_slot
    if board.is_en_passant(move):
        cap_sq = chess.square(chess.square_file(move.to_square), chess.square_rank(move.from_square))
        wps += capture_to_graveyard(sq(cap_sq), "P", slot); slot += 1
    elif board.is_capture(move):
        victim = board.piece_at(move.to_square).symbol().upper()
        wps += capture_to_graveyard(sq(move.to_square), victim, slot); slot += 1
    if move.promotion:
        # TODO: pawn to graveyard, then fetch a spare queen (or the promoted kind) to to_square.
        raise NotImplementedError("promotion sequence")
    wps += move_piece(sq(move.from_square), sq(move.to_square), kind)
    if board.is_castling(move):
        rank = chess.square_rank(move.from_square)
        kingside = chess.square_file(move.to_square) == 6
        r_from = chess.square(7 if kingside else 0, rank)
        r_to = chess.square(5 if kingside else 3, rank)
        wps += move_piece(sq(r_from), sq(r_to), "R")
    return wps + park(), slot - next_grave_slot

## Waypoints -> joint targets

IK each Cartesian waypoint (seeded from the previous solution so the arm keeps its
configuration), run the safety check on each target, and fail loudly if anything is wrong.
`densify` then splits every segment into steps no larger than `velocity.max_step_rad`
(linear in joint space) and re-checks each intermediate point, including the step size.

In [ ]:
@dataclass
class JointTarget:
    name: str
    q: np.ndarray
    gripper: float
    slow: bool

class PlanningError(RuntimeError):
    pass

# Tilts (deg from vertical) tried in order until IK converges inside the safety limits.
# Positive leans away from the arm (far ranks), negative toward it (near ranks).
TILT_CANDIDATES_DEG = [0, 10, -10, 20, -20, 30, -30, 35, -35]

def solve_waypoint(wp, seed):
    """IK for one Cartesian waypoint: (q, tilt_deg) or (None, None)."""
    lo, hi = joint_limits()
    tilts = [wp.tilt_deg] + [t for t in TILT_CANDIDATES_DEG if t != wp.tilt_deg]
    for tilt in tilts:
        if abs(tilt) > SAFETY["workspace"]["max_tilt_deg"]:
            continue
        q, info = ik_top_down(wp.xyz, tilt, q0=seed, lower=lo, upper=hi)
        if info["converged"]:
            return q, tilt
    return None, None

def plan_joint_trajectory(waypoints, q_start=None):
    q_prev = np.array(ARM["park"] if q_start is None else q_start, float)
    seed = np.array([0.0, 0.2, 0.3, 1.4, 0.0])        # bent pose above the board
    out = []
    for wp in waypoints:
        if wp.q is not None:
            q = np.asarray(wp.q, float)
            problems = validate_command(q, check_fk=False)
        else:
            q, tilt = solve_waypoint(wp, seed)
            if q is None:
                raise PlanningError(f"IK failed at {wp.name} {np.round(wp.xyz, 4).tolist()} for every tilt in {TILT_CANDIDATES_DEG}")
            problems = validate_command(q)
            seed = q
        if problems or not gripper_command_ok(wp.gripper):
            raise PlanningError(f"{wp.name}: {problems or 'gripper out of range'}")
        out.append(JointTarget(wp.name, q, wp.gripper, wp.slow))
        q_prev = q
    return out

In [ ]:
def densify(trajectory, max_step=None, q_start=None):
    """Insert joint-space linear interpolation points so no joint moves more than `max_step`
    rad between consecutive targets; every point is safety-checked (step + pose).
    TODO: Cartesian straight-line interpolation for descend/lift (`slow` segments), and timing
    (time_from_start) from velocity.max_joint_rad_s / max_tool_m_s."""
    max_step = SAFETY["velocity"]["max_step_rad"] * 0.95 if max_step is None else max_step
    q_prev = np.array(ARM["park"] if q_start is None else q_start, float)
    g_prev = ARM["gripper_open"]
    out = []
    for t in trajectory:
        n = max(1, int(np.ceil(np.abs(t.q - q_prev).max() / max_step)))
        for k in range(1, n + 1):
            q = q_prev + (t.q - q_prev) * k / n
            problems = validate_command(q, q_prev=out[-1].q if out else q_prev, check_fk=False)
            if problems:
                raise PlanningError(f"{t.name} step {k}/{n}: {problems}")
            out.append(JointTarget(t.name if k == n else f"{t.name} ({k}/{n})", q,
                                   t.gripper if k == n else g_prev, t.slow))
        q_prev, g_prev = t.q, t.gripper
    return out

### Which squares can we actually reach?

`reachability_table()` prints, per square, the tilt that works for both hover and pawn
grasp height (`--` = none). With the current board.yaml (center 0.215 m, hover 0.07 m) and
the URDF wrist_flex limit, **c1 to f1 hover is unreachable**: the arm is too folded to put the
tip 7 cm above squares that close to the base. Options (TODO, decide on the real arm): a lower
hover height for rank 1, moving the board a little farther away, or a tilted approach that
leans further back than 35 deg.

In [ ]:
def reachability_table(kind="P"):
    lines = []
    for r in range(8, 0, -1):
        cells = []
        for f in FILES:
            sq = f"{f}{r}"
            seed = np.array([0.0, 0.2, 0.3, 1.4, 0.0])
            qs = [solve_waypoint(Waypoint("t", xyz=p), seed) for p in (hover_point(sq), grasp_point(sq, kind))]
            cells.append("--" if any(q is None for q, _ in qs) else f"{max(abs(t) for _, t in qs):2d}")
        lines.append(f"{r} " + " ".join(cells))
    lines.append("  " + " ".join(f" {f}" for f in FILES))
    return "\n".join(lines)

# print(reachability_table())      # ~5 s

## Execution backends

One interface, several implementations. Only the dry run works here; the rest need hardware
or a running simulator.

In [ ]:
class Backend:
    def execute(self, trajectory):
        """Run a list of JointTarget. Return True on success."""
        raise NotImplementedError

    def gripper_position(self):
        """Measured gripper joint (rad), used by grasp verification."""
        raise NotImplementedError

@dataclass
class DryRunBackend(Backend):
    log: list = field(default_factory=list)
    gripper: float = ARM["gripper_open"]

    def execute(self, trajectory):
        for t in trajectory:
            self.log.append((t.name, np.round(t.q, 3).tolist(), t.gripper))
            self.gripper = t.gripper
        return True

    def gripper_position(self):
        return self.gripper

class ServoBackend(Backend):
    """TODO: own a ServoBus (servo_bus.ipynb) and an EStop (safety.ipynb); interpolate each
    segment at safety.yaml speeds, set_joint_targets() at ~50-100 Hz, wait until within tolerance
    of each target (read_positions), abort on e-stop."""
    def execute(self, trajectory):
        raise NotImplementedError("ServoBackend")

class GazeboBackend(Backend):
    """TODO: rclpy action client for /arm_controller/follow_joint_trajectory and
    /gripper_controller/follow_joint_trajectory (joint names in controllers.yaml), one goal per
    segment with time_from_start from the speed limits. Pieces are static in Gazebo: attach /
    teleport the piece on grasp/release (gambit_sim teleport notebook)."""
    def execute(self, trajectory):
        raise NotImplementedError("GazeboBackend")

class LeRobotBackend(Backend):
    """TODO: send joint targets through LeRobot's SO-101 follower robot interface (degrees,
    its own calibration files), so scripted and learned policies share one execution path."""
    def execute(self, trajectory):
        raise NotImplementedError("LeRobotBackend")

## Self-check

In [ ]:
wps = pick("e2", "P")
assert [w.name for w in wps] == ["hover e2", "descend", "grasp", "lift"]
assert np.isclose(wps[0].xyz[2], 0.006 + ARM["hover_height"])
assert np.isclose(wps[1].xyz[2], grasp_point("e2", "P")[2]) and wps[2].gripper == ARM["gripper_closed"]
mp = move_piece("e2", "e4", "P")
assert len(mp) == 8 and mp[-1].gripper == ARM["gripper_open"] and mp[5].gripper == ARM["gripper_closed"]
cap = capture_to_graveyard("d5", "N", 2)
assert np.allclose(cap[-1].xyz[:2], graveyard_slot(2)[:2])
traj = plan_joint_trajectory(move_piece("e2", "e4", "P") + park())
assert len(traj) == 9 and all(np.all(np.isfinite(t.q)) for t in traj)
for t, w in zip(traj[:-1], mp):
    assert np.linalg.norm(tool_position(t.q) - w.xyz) < 1e-3
dense = densify(traj)
assert len(dense) > len(traj) and dense[-1].name == "park"
steps = np.abs(np.diff(np.array([ARM["park"]] + [t.q for t in dense]), axis=0))
assert steps.max() <= SAFETY["velocity"]["max_step_rad"] + 1e-9
be = DryRunBackend(); assert be.execute(dense) and be.gripper_position() == ARM["gripper_open"]
import chess
b = chess.Board("r3k2r/pppq1ppp/2n5/3pP3/8/8/PPP2PPP/R3K2R w KQkq d6 0 1")
w_ep, used = plan_chess_move(b, chess.Move.from_uci("e5d6"))
assert used == 1 and w_ep[0].name == "hover d5"                    # en passant removes d5 pawn
w_c, used = plan_chess_move(b, chess.Move.from_uci("e1g1"))
assert used == 0 and [w.name for w in w_c].count("grasp") == 2 and w_c[-1].name == "park"
print("pick_place OK")